
### Week 7 Assignment | Celebal Technologies Data Science Internship
**Name:** Ronak Sharma




---
## Step 1 — Install Dependencies & Import Libraries

In [ ]:
import os
import re
import time
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.environ["GROQ_API_KEY"] = "_your_new__key_here"


# Core ML & Retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import faiss

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF Reading
import pypdf
from pypdf import PdfReader

# DOCX Reading
from docx import Document

# Optional: Groq
try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False

# Optional: Sentence-BERT
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    SBERT_AVAILABLE = False

print("=" * 55)
print("Library Status:")
print(f"FAISS               : v{faiss.__version__}")
print(f"Groq            : {'Installed' if GROQ_AVAILABLE else 'Not Installed'}")
print(f"Sentence-BERT       : {'Installed' if SBERT_AVAILABLE else 'Not Installed (will use TF-IDF)'}")
print(f"LangChain Splitters : Installed")
print("=" * 55)

---
## Step 2 — Build the Career Knowledge Base

This is the **document corpus** the RAG system retrieves from.
It covers 9 AI/ML roles in depth — skills, tools, salaries, hiring companies, and tips.

> **On Colab:** You can replace this with any PDF — just call `load_pdf('your_file.pdf')`.


In [ ]:
CAREER_KB = """
=== ROLE: Machine Learning Engineer ===
Machine Learning Engineer is one of the most in-demand roles in the AI industry.
Core responsibilities include designing ML pipelines, training models, and deploying them to production.
Required technical skills: Python, NumPy, Pandas, Scikit-learn, TensorFlow or PyTorch, SQL, Git.
Mathematics required: linear algebra, calculus, probability and statistics.
Soft skills: problem solving, communication, ability to explain ML results to business stakeholders.
Salary range in India: 12 to 25 LPA for freshers, 25 to 50 LPA for experienced professionals.
Top companies hiring ML Engineers: Google, Microsoft, Amazon, Flipkart, Swiggy, Razorpay, Meesho.
Typical interview rounds: coding on LeetCode, SQL queries, ML fundamentals, case study, system design.
Career path: Junior ML Engineer → Senior ML Engineer → ML Tech Lead → Director of AI.

=== ROLE: Data Scientist ===
Data Scientist bridges the gap between raw data and actionable business insights.
Key responsibilities: data collection, cleaning, exploratory data analysis, model building, storytelling.
Technical skills required: Python, R, SQL, Scikit-learn, Matplotlib, Seaborn, Tableau, Power BI.
Strong statistics background is essential — hypothesis testing, A/B testing, regression, Bayesian methods.
Domain knowledge in the industry you work in (fintech, healthcare, e-commerce) adds huge value.
Salary range in India: 8 to 20 LPA entry level, 20 to 45 LPA senior level.
Companies hiring Data Scientists: Walmart, Zomato, Paytm, CRED, McKinsey, BCG, Deloitte.
Interview process: SQL test, Python coding, statistics questions, product metrics case study.
Career progression: Data Analyst → Data Scientist → Senior Data Scientist → Principal Scientist.

=== ROLE: Deep Learning Engineer ===
Deep Learning Engineer builds and optimizes neural network architectures for complex tasks.
Responsibilities include designing CNN, RNN, LSTM, GRU, Transformer architectures.
Essential frameworks: TensorFlow, PyTorch, Keras, ONNX for model export.
GPU programming knowledge: CUDA basics, mixed precision training, distributed training.
Research reading: staying updated with arXiv papers in computer vision and NLP is expected.
Salary range in India: 15 to 35 LPA. Global companies pay 40 to 120 LPA for strong candidates.
Top employers: NVIDIA, Adobe, Samsung Research, Qualcomm, IIT research labs.
Kaggle competition wins and arXiv paper publications significantly boost the profile.
Interview includes coding, deep learning theory, paper discussions, and implementation tasks.

=== ROLE: Natural Language Processing Engineer ===
NLP Engineer specializes in making machines understand and generate human language.
Core responsibilities: text preprocessing, building language models, fine-tuning LLMs, RAG systems.
Key libraries and tools: NLTK, SpaCy, HuggingFace Transformers, LangChain, FAISS, ChromaDB.
Working with BERT, GPT, T5, LLaMA, and Mistral models for various NLP tasks.
Must understand tokenization, embeddings, attention mechanisms, and prompt engineering.
Salary range in India: 14 to 30 LPA. LLM specialists can earn 30 to 80 LPA globally.
High demand companies: Amazon Alexa, Google Search, Microsoft Copilot, Koo, ShareChat.
Interview topics: NLP fundamentals, transformer architecture, fine-tuning strategies, RAG design.
Portfolio must show: text classification, NER, summarization, or chatbot projects.

=== ROLE: Computer Vision Engineer ===
Computer Vision Engineer builds models that help machines interpret and understand visual data.
Core tasks: image classification, object detection, image segmentation, facial recognition, OCR.
Essential tools: OpenCV, YOLO (v8, v10), Detectron2, Roboflow, TensorRT for edge deployment.
Deep understanding of CNN architectures: VGG, ResNet, EfficientNet, Vision Transformers (ViT).
Applications: autonomous vehicles, medical imaging, retail checkout automation, surveillance.
Salary range: 14 to 32 LPA in India. Autonomous vehicle companies pay premium globally.
Top employers: Tesla, Waymo, Ola, Mfine, iMerit, ISRO, and various defense research labs.
Strong GitHub portfolio with annotated datasets and trained model demos is essential.
Interview includes coding, CV paper discussions, and building a small model live.

=== ROLE: MLOps Engineer ===
MLOps Engineer ensures ML models are reliably deployed, monitored and retrained in production.
Responsibilities: building CI/CD pipelines for ML, model versioning, monitoring drift, retraining.
Core tools: Docker, Kubernetes, Apache Airflow, MLflow, Weights and Biases, GitHub Actions.
Cloud platforms: AWS SageMaker, GCP Vertex AI, Azure ML for managed ML infrastructure.
Data pipeline tools: Apache Kafka, Spark, dbt, Snowflake for data engineering integration.
Salary range in India: 16 to 35 LPA. Senior MLOps roles at FAANG pay 50 to 100 LPA.
High demand because many companies have models in research but struggle to deploy at scale.
Background in DevOps, software engineering or cloud architecture is very helpful.
Interview tests: system design for ML platforms, troubleshooting production failures, Kubernetes tasks.

=== ROLE: Data Engineer ===
Data Engineer designs and builds the infrastructure that makes data available for analysis and ML.
Core responsibilities: building ETL pipelines, data warehousing, data quality monitoring.
Essential technologies: Apache Spark, Apache Kafka, Airflow, dbt, Hadoop, BigQuery, Snowflake.
Programming: Python and Scala are primary languages. SQL expertise is non-negotiable.
Cloud certifications: AWS Data Engineer, GCP Professional Data Engineer, Azure Data Engineer.
Salary range in India: 10 to 25 LPA entry level, 25 to 55 LPA senior level.
Companies hiring: Uber, Grab, PhonePe, Zepto, Dunzo, banks and large e-commerce platforms.
Career path: Junior Data Engineer → Senior → Lead → Data Architect → VP of Data Engineering.
Interview rounds: SQL complex queries, Spark optimization, system design for data pipelines.

=== ROLE: AI Research Scientist ===
AI Research Scientist advances the field by developing new algorithms and publishing papers.
Typical duties: designing experiments, running ablations, writing papers for NeurIPS, ICML, ICLR.
A PhD in Computer Science, Statistics, or Mathematics from a top university is usually required.
Strong publication record and open source contributions to major frameworks are highly valued.
Working knowledge of experiment tracking, statistical significance testing, and benchmark evaluation.
Salary range in India: 25 to 60 LPA at top labs, globally 80 to 300 LPA at OpenAI, DeepMind.
Employers: DeepMind, OpenAI, Microsoft Research, Google Brain (now Google DeepMind), FAIR (Meta).
Application requires research statement, paper portfolio, and intense technical interviews.

=== CAREER ROADMAP FOR FRESHERS ===
Step 1: Learn Python thoroughly — variables, loops, functions, OOP, file handling, libraries.
Step 2: Study NumPy, Pandas, Matplotlib and Seaborn for data manipulation and visualization.
Step 3: Learn SQL — SELECT, GROUP BY, JOINs, window functions, subqueries.
Step 4: Study Statistics — mean, median, variance, distributions, hypothesis testing, p-values.
Step 5: Learn Machine Learning with Scikit-learn — regression, classification, clustering, evaluation.
Step 6: Build 3 to 5 end-to-end projects on different domains and push them to GitHub.
Step 7: Participate in Kaggle competitions to build ranking and real-world problem solving skills.
Step 8: Learn one deep learning framework — TensorFlow or PyTorch. Build an image or text project.
Step 9: Apply for internships first — many companies convert interns to full-time employees.
Step 10: Prepare for interviews — LeetCode, SQL practice, revise ML theory and statistics.

=== RESUME AND INTERVIEW TIPS ===
Resume tips: keep it to one page for under 5 years of experience. Use bullet points with numbers.
Quantify everything — instead of 'built a model', write 'built a churn prediction model with 92% AUC'.
Mention dataset sizes, business impact, latency improvements, cost reductions wherever possible.
Use ATS-friendly formatting with standard fonts. Avoid tables and graphics in resume PDF.
Projects section should list GitHub link, tech stack, dataset, result metric and business use case.
LinkedIn profile must mirror resume. Add skills section, get endorsements, write a strong headline.

Interview preparation for coding rounds: practice LeetCode Easy and Medium questions daily for 60 days.
For SQL rounds: practice complex GROUP BY, window functions (RANK, ROW_NUMBER, LAG, LEAD), CTEs.
ML concepts to master: bias-variance tradeoff, regularization, cross-validation, feature engineering.
Evaluation metrics: accuracy, precision, recall, F1, AUC-ROC, RMSE — know when to use each.
For case study rounds: practice defining the ML problem, choosing metrics, handling class imbalance.
System design for ML: know how to design a recommendation engine, fraud detection or search ranking.
Behavioral rounds: use STAR format. Prepare stories about teamwork, failure, leadership, learning.

=== TOOLS AND CERTIFICATIONS ===
Cloud certifications that add value: AWS Solutions Architect, GCP Professional ML Engineer, Azure AI.
Course platforms: Coursera Andrew Ng ML course, fast.ai for deep learning, Kaggle Learn for free.
Community participation: attend local meetups, follow AI researchers on Twitter, join Discord servers.
Writing technical blogs on Medium or Substack increases visibility to recruiters significantly.
Contributing to popular open-source projects like Scikit-learn or HuggingFace builds credibility.
Hackathons and paper implementations posted publicly on GitHub create strong talking points.
"""

print(f"Knowledge base loaded: {len(CAREER_KB):,} characters")
print(f"Topics: ML Engineer, Data Scientist, Deep Learning, NLP, CV, MLOps, Data Eng, Research, Roadmap, Tips")


---
## Step 3 — Document Loaders

The system supports three document sources. The knowledge base above is used by default.
On Colab you can load any PDF by calling `load_pdf('your_file.pdf')`.


In [ ]:
class Document:
    """A single text passage with its source metadata."""
    def __init__(self, content, metadata=None):
        self.content  = content.strip()
        self.metadata = metadata or {}

    def __repr__(self):
        return f"Document(source='{self.metadata.get('source','?')}', chars={len(self.content)})"


def load_pdf(pdf_path):
    """Load a PDF file and return one Document per page."""
    reader = pypdf.PdfReader(pdf_path)
    docs = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            docs.append(Document(text, {'source': os.path.basename(pdf_path), 'page': i+1}))
    print(f"PDF loaded: {len(docs)} pages from '{pdf_path}'")
    return docs


def load_text_file(path):
    """Load a plain text file."""
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    print(f"Text file loaded: {len(text):,} chars from '{path}'")
    return [Document(text, {'source': os.path.basename(path)})]


def load_raw_string(text, name='knowledge_base'):
    """Use any string directly as a document."""
    doc = Document(text, {'source': name})
    print(f"Raw text loaded: {len(text):,} chars from '{name}'")
    return [doc]


# Load the career knowledge base
raw_documents = load_raw_string(CAREER_KB, name='AI_Career_Knowledge_Base')



---
## Step 4 — Text Chunking

We split the knowledge base into overlapping passages.
**Chunk size = 300 characters, overlap = 50** — tuned for career Q&A where each chunk should cover one complete idea (a role's skills, a salary range, one career step).


In [ ]:
CHUNK_SIZE    = 300
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separators    = ["\n\n", "\n", ". ", " ", ""]
)


def chunk_documents(docs):
    """Split each Document into overlapping chunks. Returns list of Documents."""
    chunks = []
    for doc in docs:
        sub_texts = splitter.split_text(doc.content)
        for idx, text in enumerate(sub_texts):
            if text.strip():
                chunks.append(Document(
                    content  = text.strip(),
                    metadata = {**doc.metadata, 'chunk_id': idx, 'chunk_total': len(sub_texts)}
                ))
    return chunks


all_chunks = chunk_documents(raw_documents)
chunk_texts = [c.content for c in all_chunks]

print(f"Chunking params : size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"Total chunks    : {len(all_chunks)}")
lengths = [len(c.content) for c in all_chunks]
print(f"Length — min: {min(lengths)}  max: {max(lengths)}  mean: {np.mean(lengths):.0f}")
print()
print("Sample chunks:")
for i in [0, 5, 10, 20]:
    print(f"  [{i}] {all_chunks[i].content[:100]}...")


In [ ]:
# Chunk length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(lengths, bins=20, color='steelblue', edgecolor='white', alpha=0.9)
axes[0].axvline(np.mean(lengths), color='tomato', lw=2, ls='--',
                label=f'Mean = {np.mean(lengths):.0f}')
axes[0].axvline(CHUNK_SIZE, color='seagreen', lw=1.5, ls=':', label=f'Target = {CHUNK_SIZE}')
axes[0].set_title('Chunk Length Distribution', fontweight='bold')
axes[0].set_xlabel('Characters per Chunk'); axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].bar(range(len(all_chunks)), lengths, color='steelblue', alpha=0.8, edgecolor='none')
axes[1].axhline(CHUNK_SIZE, color='tomato', lw=1.5, ls='--', label=f'Target {CHUNK_SIZE}')
axes[1].set_title('Each Chunk Size', fontweight='bold')
axes[1].set_xlabel('Chunk Index'); axes[1].set_ylabel('Characters')
axes[1].legend()

plt.suptitle(f'Text Chunking — {len(all_chunks)} chunks from knowledge base',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('step4_chunks.png', bbox_inches='tight', dpi=110)
plt.show()


---
## Step 5 — Embedding Creation

Embeddings convert text into numerical vectors. Texts with similar meaning end up with vectors that are close together — this is what enables semantic retrieval.




In [ ]:
class TFIDFEmbedder:
    """
    TF-IDF sparse embedder — works fully offline.
    Best when query and document share exact keywords.
    n-gram range (1,2) captures single words AND two-word phrases.
    """
    def __init__(self, max_features=8000):
        self.vectorizer = TfidfVectorizer(
            max_features = max_features,
            ngram_range  = (1, 2),
            stop_words   = 'english',
            sublinear_tf = True
        )
        self.fitted = False

    def fit(self, texts):
        self.vectorizer.fit(texts)
        self.fitted = True
        print(f"TF-IDF fitted | vocab: {len(self.vectorizer.vocabulary_):,} | "
              f"corpus: {len(texts)} chunks")

    def embed(self, texts):
        if isinstance(texts, str): texts = [texts]
        return self.vectorizer.transform(texts).toarray().astype(np.float32)

    @property
    def dim(self):
        return len(self.vectorizer.vocabulary_)


class SBERTEmbedder:
    """
    Sentence-BERT dense embedder — downloads model on first run (~90MB).
    Much better for semantic similarity and paraphrased queries.
    """
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        print(f"Loading SBERT model: {model_name}...")
        self.model  = SentenceTransformer(model_name)
        self.fitted = True
        print(f"SBERT ready | embedding dim: {self.model.get_sentence_embedding_dimension()}")

    def fit(self, texts):
        pass  # SBERT is pretrained, no fitting needed

    def embed(self, texts):
        if isinstance(texts, str): texts = [texts]
        return self.model.encode(texts, show_progress_bar=False,
                                  convert_to_numpy=True).astype(np.float32)

    @property
    def dim(self):
        return self.model.get_sentence_embedding_dimension()


# ── Choose embedder ──────────────────────────────────────────────────────────
if SBERT_AVAILABLE:
    try:
        embedder = SBERTEmbedder()
        EMBED_TYPE = 'SBERT (dense)'
    except Exception as e:
        print(f"SBERT failed ({e}), falling back to TF-IDF")
        embedder = TFIDFEmbedder()
        embedder.fit(chunk_texts)
        EMBED_TYPE = 'TF-IDF (sparse)'
else:
    embedder = TFIDFEmbedder()
    embedder.fit(chunk_texts)
    EMBED_TYPE = 'TF-IDF (sparse)'

print(f"\nEmbedder in use : {EMBED_TYPE}")

# Embed all chunks
chunk_embeddings = embedder.embed(chunk_texts)
print(f"Embedding matrix: {chunk_embeddings.shape}  (chunks × features)")


### 5.1 Visualise Embedding Space with PCA

In [ ]:
pca2d = PCA(n_components=2, random_state=42)
embs_2d = pca2d.fit_transform(chunk_embeddings)

# Auto-label chunks by detected topic
TOPIC_KEYWORDS = {
    'ML Engineer':     ['machine learning engineer', 'ml engineer', 'scikit-learn'],
    'Data Scientist':  ['data scientist', 'data science', 'storytelling'],
    'Deep Learning':   ['deep learning', 'cnn', 'rnn', 'lstm', 'gpu'],
    'NLP Engineer':    ['nlp', 'language', 'bert', 'gpt', 'transformer'],
    'CV Engineer':     ['computer vision', 'opencv', 'yolo', 'image'],
    'MLOps':           ['mlops', 'docker', 'kubernetes', 'airflow'],
    'Data Engineer':   ['data engineer', 'spark', 'kafka', 'pipeline'],
    'Research/PhD':    ['research', 'phd', 'paper', 'publication'],
    'Roadmap/Tips':    ['fresher', 'resume', 'interview', 'roadmap', 'step'],
}

def label_chunk(text):
    t = text.lower()
    for topic, kws in TOPIC_KEYWORDS.items():
        if any(kw in t for kw in kws):
            return topic
    return 'General'

chunk_labels = [label_chunk(c.content) for c in all_chunks]
unique_labels = sorted(set(chunk_labels))
palette       = plt.cm.tab10(np.linspace(0, 0.9, len(unique_labels)))
color_map     = dict(zip(unique_labels, palette))

plt.figure(figsize=(12, 8))
for label in unique_labels:
    mask = [l == label for l in chunk_labels]
    pts  = embs_2d[mask]
    plt.scatter(pts[:, 0], pts[:, 1],
                c=[color_map[label]], label=label,
                s=90, alpha=0.85, edgecolors='white', lw=0.5)

plt.title(f'Knowledge Base — Chunk Embeddings (PCA 2D)\n'
          f'PC1={pca2d.explained_variance_ratio_[0]*100:.1f}%  '
          f'PC2={pca2d.explained_variance_ratio_[1]*100:.1f}%  |  '
          f'Embedder: {EMBED_TYPE}',
          fontsize=12, fontweight='bold')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, framealpha=0.9)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('step5_embeddings_pca.png', bbox_inches='tight', dpi=110)
plt.show()
print("Clusters confirm that similar role information groups together in the embedding space.")


---
## Step 6 — FAISS Vector Store

FAISS (Facebook AI Similarity Search) stores all chunk embeddings and answers
"which chunks are most similar to this query?" in milliseconds — even for millions of vectors.

We use **IndexFlatIP** (Inner Product on L2-normalised vectors = cosine similarity).


In [ ]:
class CareerVectorStore:
    """
    FAISS-backed vector store for the career knowledge base.
    Stores chunk embeddings and supports fast top-k cosine similarity retrieval.
    """

    def __init__(self, embedder):
        self.embedder  = embedder
        self.documents = []
        self.index     = None
        self._norm_embs = None

    def build(self, documents, precomputed_embeddings=None):
        """Index all documents. Reuses precomputed embeddings if provided."""
        self.documents = documents
        texts = [d.content for d in documents]

        embs = precomputed_embeddings if precomputed_embeddings is not None                else self.embedder.embed(texts)

        # L2-normalise → inner product becomes cosine similarity
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        norms = np.where(norms < 1e-10, 1e-10, norms)
        normed = (embs / norms).astype(np.float32)

        self.index      = faiss.IndexFlatIP(normed.shape[1])
        self.index.add(normed)
        self._norm_embs = normed

        print(f"Vector store built: {self.index.ntotal} vectors | dim={normed.shape[1]}")

    def search(self, query, top_k=4):
        """Return top-k most relevant Document objects with their scores."""
        q_emb  = self.embedder.embed([query]).astype(np.float32)
        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-10)

        scores, indices = self.index.search(q_norm, min(top_k, self.index.ntotal))

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0:
                doc = self.documents[idx]
                results.append({
                    'document': doc,
                    'score':    float(score),
                    'source':   doc.metadata.get('source', 'KB'),
                    'chunk_id': doc.metadata.get('chunk_id', idx),
                })
        return results

    def __len__(self):
        return len(self.documents)


# Build the career vector store
career_store = CareerVectorStore(embedder)
career_store.build(all_chunks, precomputed_embeddings=chunk_embeddings)
print(f"\nVector store ready: {len(career_store)} career knowledge chunks indexed")


---
## Step 7 — Answer Generation

The generator builds a context-aware prompt and produces an answer.




In [ ]:
def build_prompt(question, retrieved_chunks):
    """Assemble the RAG prompt: system instruction + retrieved context + question."""
    context_blocks = []
    for i, r in enumerate(retrieved_chunks):
        block = f"[Source {i+1} | {r['source']} | relevance={r['score']:.3f}]\n{r['document'].content}"
        context_blocks.append(block)
    context_text = "\n\n".join(context_blocks)

    prompt = f"""You are an expert AI Career Counselor. Answer the candidate's question
using ONLY the information in the provided context passages.
Be specific, practical, and encourage the candidate.
If the context does not contain the answer, say so honestly.

--- CONTEXT ---
{context_text}
--- END CONTEXT ---

Candidate's Question: {question}

Instructions:
- Give a clear, well-structured answer in 3-6 sentences
- Include specific skills, tools, salaries, or steps wherever they appear in the context
- End with one practical next-step recommendation
- Reference which source passages you used

Answer:""".strip()
    return prompt


class GroqCareerAdvisor:
    """
    Uses Groq  (llama-3.1-8b-instant) for fast, high-quality career advice.
    Free tier available at https://console.groq.com
    Set _KEY environment variable before using this.
    """
    def __init__(self, model='llama-3.1-8b-instant', max_tokens=600):
        self.model      = model
        self.max_tokens = max_tokens
        api_key         = os.environ.get('GROQ__KEY', '')
        if not api_key:
            raise ValueError(
                "GROQ_API_KEY not set.\n"
                "Get a free key at https://console.groq.com\n"
                "Then run: os.environ['GROQ_API_KEY'] = '_your_key_here'"
            )
        self.client = Groq(api_key=api_key)

    def generate(self, question, retrieved):
        prompt = build_prompt(question, retrieved)
        response = self.client.chat.completions.create(
            model    = self.model,
            messages = [
                {
                    "role":    "system",
                    "content": "You are an expert AI Career Counselor who gives precise, "
                               "grounded advice based only on the provided context."
                },
                {
                    "role":    "user",
                    "content": prompt
                }
            ],
            max_tokens  = self.max_tokens,
            temperature = 0.3,   # lower temperature = more factual, less hallucination
        )
        return response.choices[0].message.content.strip()


class ExtractiveCounselor:
    """
    Offline extractive answer generator — no API key needed.
    Splits retrieved chunks into sentences, re-ranks by query similarity,
    and returns the top 4 most relevant sentences as the answer.
    """
    def __init__(self, embedder):
        self.embedder = embedder

    def generate(self, question, retrieved):
        sentences, meta = [], []
        for r in retrieved:
            raw_sents = re.split(r'(?<=[.?!])\s+', r['document'].content)
            for s in raw_sents:
                s = s.strip()
                if len(s) > 50:
                    sentences.append(s)
                    meta.append(r['source'])

        if not sentences:
            return "I could not find relevant information in the knowledge base for this question."

        q_emb   = self.embedder.embed([question]).astype(np.float32)
        s_embs  = self.embedder.embed(sentences).astype(np.float32)
        sims    = cosine_similarity(q_emb, s_embs)[0]
        top_idx = np.argsort(sims)[::-1][:4]

        chosen = [sentences[i] for i in top_idx if sims[i] > 0.02]
        if not chosen:
            return "The knowledge base does not contain a direct answer to this question."

        return ' '.join(chosen)


# ── Initialise the generator 
GROQ_KEY = os.environ.get('GROQ_API_KEY', '')

if GROQ_AVAILABLE and GROQ_KEY:
    try:
        generator = GroqCareerAdvisor()
        GEN_TYPE  = f'Groq API (llama-3.1-8b-instant)'
        print(f"✅ Generator: {GEN_TYPE}")
    except Exception as e:
        print(f"⚠️  Groq init failed: {e}")
        generator = ExtractiveCounselor(embedder)
        GEN_TYPE  = 'Extractive (offline fallback)'
        print(f"Generator: {GEN_TYPE}")
else:
    generator = ExtractiveCounselor(embedder)
    GEN_TYPE  = 'Extractive (offline — set GROQ_API_KEY for Groq LLM)'
    print(f"Generator: {GEN_TYPE}")

print()
print("─" * 50)
print("To enable Groq LLM (FREE):")
print("  1. Get key at https://console.groq.com")
print("  2. import os")
print("     os.environ['GROQ_API_KEY'] = '_your_key_here'")
print("  3. generator = GroqCareerAdvisor()")
print("     rag = CareerRAG(store=career_store, generator=generator)")
print("─" * 50)


---
## Step 8 — Hybrid Search (Keyword + Semantic)

Hybrid search blends two signals:
- **Semantic score** (FAISS cosine similarity) — captures meaning and context
- **BM25 keyword score** — rewards exact term matches like 'MLOps', 'LSTM', 'LPA'

This makes the system more robust — semantic alone misses acronyms, keyword alone misses paraphrasing.


In [ ]:
def bm25_score(query, document_text, k1=1.5, b=0.75, avg_doc_len=50):
    """Simplified BM25 term-frequency scoring."""
    qterms   = query.lower().split()
    dwords   = document_text.lower().split()
    doc_len  = len(dwords)
    score    = 0.0
    for term in qterms:
        tf = dwords.count(term)
        if tf > 0:
            score += (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avg_doc_len))
    return score


def hybrid_search(query, store, top_k=4, alpha=0.65):
    """
    alpha * semantic_score + (1-alpha) * bm25_score → final rank.
    alpha=0.65 gives slight preference to semantic similarity.
    """
    # Semantic retrieval (get all scores)
    sem_results = store.search(query, top_k=len(store.documents))
    sem_map = {r['chunk_id']: r for r in sem_results}
    sem_scores = {r['chunk_id']: r['score'] for r in sem_results}

    # BM25 scores for every chunk
    bm25_raw = {}
    for i, doc in enumerate(store.documents):
        cid = doc.metadata.get('chunk_id', i)
        bm25_raw[cid] = bm25_score(query, doc.content)

    max_bm = max(bm25_raw.values(), default=1.0)
    bm25_n = {k: v / (max_bm + 1e-10) for k, v in bm25_raw.items()}

    # Combine scores
    all_ids  = set(sem_scores) | set(bm25_n)
    combined = {
        cid: alpha * sem_scores.get(cid, 0.0) + (1 - alpha) * bm25_n.get(cid, 0.0)
        for cid in all_ids
    }

    top_ids = sorted(combined, key=lambda c: combined[c], reverse=True)[:top_k]
    return [
        {**sem_map[cid],
         'hybrid_score':   combined[cid],
         'semantic_score': sem_scores.get(cid, 0),
         'bm25_score':     bm25_n.get(cid, 0)}
        for cid in top_ids if cid in sem_map
    ]

print("Hybrid search function ready (alpha=0.65 semantic + 0.35 BM25 keyword)")


---
## Step 9 — Complete RAG Pipeline Class

This ties everything together into a single `CareerRAG` object.
Call `.ask(question)` and get a grounded answer with sources.


In [ ]:
class CareerRAG:
    """
    End-to-end AI Career Counselor RAG system.

    Usage:
        rag = CareerRAG(store=career_store, generator=generator, top_k=4)
        result = rag.ask("What skills do I need to become an ML Engineer?")
    """

    def __init__(self, store, generator, top_k=4, use_hybrid=True):
        self.store       = store
        self.generator   = generator
        self.top_k       = top_k
        self.use_hybrid  = use_hybrid
        self.history     = []   # keeps track of all Q&A pairs

    def ask(self, question, verbose=True):
        """Run the full pipeline: retrieve → generate → return structured result."""
        t0 = time.time()

        # ── Retrieval ────────────────────────────────────────────────────────
        if self.use_hybrid:
            retrieved = hybrid_search(question, self.store, top_k=self.top_k)
        else:
            retrieved = self.store.search(question, top_k=self.top_k)

        # ── Generation ───────────────────────────────────────────────────────
        answer  = self.generator.generate(question, retrieved)
        latency = round(time.time() - t0, 3)

        result = {
            'question':   question,
            'answer':     answer,
            'retrieved':  retrieved,
            'latency_s':  latency,
            'top_score':  retrieved[0].get('hybrid_score', retrieved[0]['score']) if retrieved else 0,
            'n_sources':  len(retrieved),
        }
        self.history.append(result)

        if verbose:
            print("\n" + "═" * 68)
            print(f" 🎓 CAREER COUNSELOR")
            print("═" * 68)
            print(f" ❓ Q: {question}")
            print("─" * 68)
            print(f" 💬 A: {answer}")
            print("─" * 68)
            print(f" 📎 Retrieved {len(retrieved)} chunks | latency={latency}s | "
                  f"top_score={result['top_score']:.3f}")
            for i, r in enumerate(retrieved[:3]):
                s = r.get('hybrid_score', r['score'])
                print(f"    [{i+1}] score={s:.3f} → {r['document'].content[:90]}...")
            print("═" * 68)

        return result

    def batch_ask(self, questions, verbose=False):
        return [self.ask(q, verbose=verbose) for q in questions]


# Build the career counselor
rag = CareerRAG(store=career_store, generator=generator, top_k=4, use_hybrid=True)
print("✅ AI Career Counselor RAG system is ready!")
print()
print("Usage: rag.ask('Your career question here')")


---
## Step 10 — Live Q&A Demo

Testing the career counselor with 12 realistic candidate questions across all topics.


In [ ]:
DEMO_QUESTIONS = [
    # Role-specific skill questions
    "What technical skills do I need to become a Machine Learning Engineer?",
    "What tools does an MLOps Engineer work with?",
    "What programming languages are important for a Data Engineer?",
    "How is an NLP Engineer different from a regular ML Engineer?",

    # Salary questions
    "What is the salary range for a Data Scientist in India?",
    "How much does a Deep Learning Engineer earn?",

    # Career roadmap questions
    "I am a fresher who wants to enter Data Science. What steps should I follow?",
    "What should I put in my resume to get a Data Science job?",

    # Interview prep questions
    "How do I prepare for the coding round of an ML Engineer interview?",
    "What ML concepts must I know before attending any AI interview?",

    # Comparison questions
    "What is the difference between a Data Scientist and a Machine Learning Engineer?",
    "Which role has the highest salary — ML Engineer, MLOps Engineer, or AI Research Scientist?",
]

print(f"Running {len(DEMO_QUESTIONS)} questions through the Career Counselor RAG...\n")
all_results = rag.batch_ask(DEMO_QUESTIONS, verbose=False)

# Print compact results
for i, r in enumerate(all_results):
    print(f"{'─'*70}")
    print(f"Q{i+1:02d}: {r['question']}")
    print(f"  A : {r['answer'][:350]}{'...' if len(r['answer'])>350 else ''}")
    print(f"      [score={r['top_score']:.3f} | latency={r['latency_s']}s]")


### 10.1 — Detailed Single Question Trace

In [ ]:
# Show one question in full detail — all retrieved chunks visible
detailed = rag.ask(
    "What are the exact steps I should follow as a fresher to get a job in AI?",
    verbose=True
)


---
## Step 11 — Evaluation Dashboard

Measuring the system's performance across all 12 demo questions.


In [ ]:
# Build evaluation dataframe
eval_rows = []
for r in all_results:
    q_lower     = r['question'].lower()
    q_words     = set(re.findall(r'\b\w{4,}\b', q_lower))
    ctx_text    = ' '.join([x['document'].content for x in r['retrieved']]).lower()
    ctx_words   = set(ctx_text.split())
    kw_coverage = len(q_words & ctx_words) / max(len(q_words), 1)

    eval_rows.append({
        'Q#':              f"Q{len(eval_rows)+1:02d}",
        'Question (short)':r['question'][:45] + '...',
        'Top Score':       round(r['top_score'], 4),
        'KW Coverage':     round(kw_coverage, 3),
        'Answer Len':      len(r['answer']),
        'Latency (s)':     r['latency_s'],
        'N Sources':       r['n_sources'],
    })

eval_df = pd.DataFrame(eval_rows)
print("="*75)
print("EVALUATION RESULTS — AI Career Counselor RAG")
print("="*75)
print(eval_df.to_string(index=False))
print()
print(f"Average top score   : {eval_df['Top Score'].mean():.4f}")
print(f"Average KW coverage : {eval_df['KW Coverage'].mean():.3f}")
print(f"Average latency     : {eval_df['Latency (s)'].mean():.3f}s")
print(f"Average answer len  : {eval_df['Answer Len'].mean():.0f} chars")


In [ ]:
# ── Evaluation visualisation ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

q_labels = eval_df['Q#'].tolist()

# 1. Retrieval score per question
bar_colors = ['seagreen' if s >= 0.2 else 'darkorange' if s >= 0.1 else 'tomato'
              for s in eval_df['Top Score']]
axes[0,0].barh(q_labels[::-1], eval_df['Top Score'][::-1],
               color=bar_colors[::-1], edgecolor='white', alpha=0.9)
axes[0,0].axvline(eval_df['Top Score'].mean(), color='black', lw=1.5, ls='--',
                  label=f"Mean={eval_df['Top Score'].mean():.3f}")
axes[0,0].set_title('Retrieval Score per Question', fontweight='bold')
axes[0,0].set_xlabel('Hybrid Score (cosine + BM25)')
axes[0,0].legend(fontsize=9)

# 2. Keyword coverage
kw_colors = ['seagreen' if v >= 0.4 else 'darkorange' if v >= 0.2 else 'tomato'
             for v in eval_df['KW Coverage']]
axes[0,1].bar(q_labels, eval_df['KW Coverage'],
              color=kw_colors, edgecolor='white', alpha=0.9)
axes[0,1].axhline(eval_df['KW Coverage'].mean(), color='black', lw=1.5, ls='--',
                  label=f"Mean={eval_df['KW Coverage'].mean():.3f}")
axes[0,1].set_title('Keyword Coverage in Retrieved Chunks', fontweight='bold')
axes[0,1].set_ylabel('Coverage (0–1)'); axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].legend(fontsize=9)
good_p  = mpatches.Patch(color='seagreen',   label='Good ≥0.4')
ok_p    = mpatches.Patch(color='darkorange', label='OK 0.2–0.4')
low_p   = mpatches.Patch(color='tomato',     label='Low <0.2')
axes[0,1].legend(handles=[good_p, ok_p, low_p], fontsize=8)

# 3. Latency
axes[1,0].bar(q_labels, eval_df['Latency (s)'],
              color='steelblue', edgecolor='white', alpha=0.9)
axes[1,0].axhline(eval_df['Latency (s)'].mean(), color='tomato', lw=1.5, ls='--',
                  label=f"Mean={eval_df['Latency (s)'].mean():.3f}s")
axes[1,0].set_title('Response Latency per Question', fontweight='bold')
axes[1,0].set_ylabel('Seconds'); axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].legend(fontsize=9)

# 4. Score vs Coverage scatter
sc = axes[1,1].scatter(eval_df['Top Score'], eval_df['KW Coverage'],
                       c=eval_df['Latency (s)'], cmap='coolwarm',
                       s=120, alpha=0.85, edgecolors='white', lw=1)
for _, row in eval_df.iterrows():
    axes[1,1].annotate(row['Q#'],
                       (row['Top Score'], row['KW Coverage']),
                       fontsize=7, ha='center', va='bottom')
plt.colorbar(sc, ax=axes[1,1], label='Latency (s)')
axes[1,1].set_title('Retrieval Score vs Keyword Coverage', fontweight='bold')
axes[1,1].set_xlabel('Top Score'); axes[1,1].set_ylabel('KW Coverage')

plt.suptitle('AI Career Counselor RAG — Evaluation Dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('step11_evaluation.png', bbox_inches='tight', dpi=110)
plt.show()


---
## Step 12 — Hybrid vs Pure Vector Search Comparison

Showing how hybrid search improves retrieval for technical acronym-heavy queries.


In [ ]:
comparison_qs = [
    "MLOps tools Docker Kubernetes Airflow",
    "salary LPA fresher data scientist",
    "BERT GPT fine-tuning NLP engineer",
    "CUDA GPU TensorRT deep learning deployment",
]

print("Comparing Semantic-only vs Hybrid retrieval:")
print()
for q in comparison_qs:
    sem_results = career_store.search(q, top_k=1)
    hyb_results = hybrid_search(q, career_store, top_k=1)

    sem_score = sem_results[0]['score']    if sem_results else 0
    hyb_score = hyb_results[0]['hybrid_score'] if hyb_results else 0
    winner    = '🟢 Hybrid wins' if hyb_score > sem_score else '⚪ Equal'

    print(f"Q: '{q}'")
    print(f"  Semantic : {sem_score:.4f} → {sem_results[0]['document'].content[:70]}...")
    print(f"  Hybrid   : {hyb_score:.4f} → {hyb_results[0]['document'].content[:70]}...  {winner}")
    print()


---
## Step 13 — Chunking Strategy Comparison

Testing three chunking approaches on the same query to find which produces the best retrieval.


In [ ]:
test_q_chunk = "What salary does an ML Engineer earn in India?"

chunk_strategies = {
    'Small (200/30)':   RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30),
    'Medium (300/50)':  RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50),
    'Large  (500/80)':  RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80),
}

strat_results = []
for name, strat in chunk_strategies.items():
    strat_chunks = strat.split_text(CAREER_KB)
    strat_docs   = [Document(t, {'source': 'KB', 'chunk_id': i})
                    for i, t in enumerate(strat_chunks) if t.strip()]
    strat_embs   = embedder.embed([d.content for d in strat_docs])

    tmp_store = CareerVectorStore(embedder)
    tmp_store.build(strat_docs, precomputed_embeddings=strat_embs)
    top = tmp_store.search(test_q_chunk, top_k=1)
    top_score = top[0]['score'] if top else 0
    top_text  = top[0]['document'].content[:80] if top else ''

    strat_results.append({
        'Strategy':    name,
        'N Chunks':    len(strat_docs),
        'Mean Len':    int(np.mean([len(d.content) for d in strat_docs])),
        'Top Score':   round(top_score, 4),
        'Top Result':  top_text + '...'
    })

cmp_df = pd.DataFrame(strat_results)
print(f"Query: '{test_q_chunk}'\n")
print(cmp_df[['Strategy','N Chunks','Mean Len','Top Score']].to_string(index=False))
print()
for row in strat_results:
    print(f"  {row['Strategy']}: {row['Top Result']}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(cmp_df['Strategy'], cmp_df['N Chunks'],   color='steelblue', edgecolor='white')
axes[0].set_title('Number of Chunks');    axes[0].tick_params(axis='x', rotation=10)
axes[1].bar(cmp_df['Strategy'], cmp_df['Mean Len'],   color='darkorange', edgecolor='white')
axes[1].set_title('Mean Chunk Length');   axes[1].tick_params(axis='x', rotation=10)
axes[2].bar(cmp_df['Strategy'], cmp_df['Top Score'],  color='seagreen', edgecolor='white')
axes[2].set_title('Top-1 Retrieval Score'); axes[2].tick_params(axis='x', rotation=10)
plt.suptitle('Chunking Strategy Comparison', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('step13_chunking.png', bbox_inches='tight', dpi=110)
plt.show()


---
## Step 14 — Load Your Own PDF

Replace the career knowledge base with any PDF of your choice.
The exact same pipeline runs — just swap the document source.


In [ ]:
# ── Try to load the assignment PDF from the working directory ────────────────
PDF_PATH = 'Week7_Project.pdf'

if os.path.exists(PDF_PATH):
    pdf_docs   = load_pdf(PDF_PATH)
    pdf_chunks = chunk_documents(pdf_docs)
    pdf_embs   = embedder.embed([c.content for c in pdf_chunks])

    pdf_store  = CareerVectorStore(embedder)
    pdf_store.build(pdf_chunks, precomputed_embeddings=pdf_embs)

    pdf_rag = CareerRAG(store=pdf_store, generator=generator, top_k=3)
    print(f"PDF RAG ready: {len(pdf_chunks)} chunks from {PDF_PATH}")
    print()
    pdf_rag.ask("What is the main objective of this project?")
    pdf_rag.ask("What are the 7 stages of the pipeline described in the document?")
    pdf_rag.ask("What improvements are suggested for the RAG system?")
else:
    print(f"'{PDF_PATH}' not found in working directory.")
    print()
    print("To use your own PDF on Colab:")
    print("  1. Upload the PDF using the file sidebar")
    print("  2. pdf_docs   = load_pdf('your_file.pdf')")
    print("  3. pdf_chunks = chunk_documents(pdf_docs)")
    print("  4. pdf_embs   = embedder.embed([c.content for c in pdf_chunks])")
    print("  5. pdf_store  = CareerVectorStore(embedder)")
    print("  6. pdf_store.build(pdf_chunks, precomputed_embeddings=pdf_embs)")
    print("  7. pdf_rag    = CareerRAG(store=pdf_store, generator=generator)")
    print("  8. pdf_rag.ask('Your question?')")


---
## Step 15 — Interactive Career Counselor

Your personal AI career counselor — just call `counsel(question)` with any career question.


In [ ]:
def counsel(question):
    """
    Ask the AI Career Counselor any question.

    Examples:
        counsel("Should I learn TensorFlow or PyTorch?")
        counsel("What certifications help for a cloud ML job?")
        counsel("How do I negotiate my salary as a Data Scientist?")
        counsel("What is the difference between MLOps and Data Engineering?")
    """
    return rag.ask(question, verbose=True)


# ── Try these questions — or write your own ──────────────────────────────────
counsel("Should I focus on becoming a Data Scientist or an ML Engineer as a fresher?")


In [ ]:
counsel("What certifications should I do to get a cloud-based ML job?")


In [ ]:
counsel("How do I build a strong GitHub portfolio to impress recruiters?")


In [ ]:
# ── Write your own question here ─────────────────────────────────────────────
# counsel("Your question here")


---
## Step 17 — System Metrics Report

In [ ]:
print("╔" + "═"*65 + "╗")
print("║   AI CAREER COUNSELOR — RAG SYSTEM METRICS REPORT              ║")
print("╠" + "═"*65 + "╣")
print(f"║  Task               : Career Q&A (skills, salary, roadmap, tips)  ║")
print(f"║  Knowledge base     : {len(CAREER_KB):,} chars, 9 AI/ML roles              ║")
print(f"║  Chunking           : Recursive, size=300, overlap=50             ║")
print(f"║  Total chunks       : {len(all_chunks)} indexed                             ║")
print(f"║  Embedding model    : {EMBED_TYPE:<42} ║")
print(f"║  Embedding dim      : {chunk_embeddings.shape[1]:,}                                   ║")
print(f"║  Vector store       : FAISS IndexFlatIP (cosine similarity)       ║")
print(f"║  Search strategy    : Hybrid (65% semantic + 35% BM25 keyword)    ║")
print(f"║  Generator          : {GEN_TYPE[:45]:<45} ║")
print(f"║  Top-k retrieval    : 4 chunks per query                          ║")
print("╠" + "═"*65 + "╣")
print(f"║  Demo questions     : {len(all_results)}                                           ║")
print(f"║  Avg retrieval score: {eval_df['Top Score'].mean():.4f}                                 ║")
print(f"║  Avg KW coverage    : {eval_df['KW Coverage'].mean():.3f}                                  ║")
print(f"║  Avg latency        : {eval_df['Latency (s)'].mean():.3f}s per query                       ║")
print(f"║  Avg answer length  : {eval_df['Answer Len'].mean():.0f} characters                     ║")
print("╚" + "═"*65 + "╝")


---
## Step 18 — Key Learnings & Conclusion

### What We Built
A fully working **AI Career Counselor** using a 7-stage RAG pipeline. The system answers
specific, grounded questions about AI/ML career paths — skills, salaries, interview prep,
roadmaps — by retrieving from a curated knowledge base rather than generating from memory.



### Key Learnings

**1. RAG grounds answers in real documents** — unlike a standalone LLM, every answer
from this system is traceable to a specific chunk in the knowledge base with a similarity
score. If the KB doesn't contain the answer, the system says so honestly.

**2. Chunking quality determines retrieval quality** — chunks that are too long average
over multiple topics, making it hard to retrieve the right passage. Chunks that are too
short lose context. 300 characters with 50-character overlap works well for fact-dense
career information.

**3. Hybrid search outperforms pure vector search** — for technical queries with specific
acronyms like 'MLOps', 'CUDA', or 'BM25', exact keyword matching through BM25 catches
what semantic similarity misses. Combining both (65/35 split) gives the best of both worlds.

**4. Embedding choice matters** — TF-IDF works well for keyword-rich domains like career
advice where query and document share vocabulary. SBERT adds semantic understanding
(knowing 'wage' and 'salary' are related) but requires an internet download.

**5. The pipeline is domain-agnostic** — swapping `CAREER_KB` for any other domain text
(legal, medical, technical) immediately gives you a domain-specific QA assistant with
zero code changes needed.
